<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@700&display=swap" rel="stylesheet">

<h1 style="
font-family: 'Oswald', sans-serif;
font-weight: 700;
font-style: italic;
font-size: 90px;
letter-spacing: 2px;
color: #E7C173;
text-shadow: 3px 3px 0 #333;
">
MACHINE LEARNING<br>IN INDUSTRY
</h1>

# Day 1: First Look at the Data

**Part 1** -- Understand the data we are given: identify common data issues before any modeling.
**Part 2** -- Prepare data for modeling, add features (covered in the second notebook).

## Table of Contents

**Setup**
- [Dataset Overview](#dataset-overview)

**Data Issues**
1. [Duplicates](#duplicates)
2. [Impossible Values](#impossible-values)
3. [Mixed Types](#mixed-types)
4. [Row-wise Column Misalignment](#row-wise-column-misalignment)
5. [Inconsistent Categorical Labels](#inconsistent-categorical-labels)
6. [Missing Values](#missing-values)
7. [Perfect Correlation / Anti-Correlation](#perfect-correlation-anti-correlation)
8. [Missing Values Correlation](#missing-values-correlation)
9. [Rare Categories](#rare-categories)
10. [Unit Inconsistencies](#unit-inconsistencies)
11. [Truncated / Capped Values](#truncated-capped-values)
12. [Implicit Hierarchies](#implicit-hierarchies)
13. [Unseen Categories at Testing Time](#unseen-categories-at-testing-time)
14. [Sparsity](#sparsity)
15. [Data Drift](#data-drift)
16. [Time Misalignment / Leakage](#time-misalignment-leakage)
17. [Target Leakage](#target-leakage)
18. [Business Process Artifacts](#business-process-artifacts)
19. [Sampling Bias / Selection Bias](#sampling-bias-selection-bias)

<details>
<summary><b>Full list of data issues covered in this notebook (click to expand)</b></summary>

| # | Issue | Example |
|---|-------|---------|
| 1 | Unseen categories at test time | New occupation values the model never saw during training |
| 2 | Perfect correlation / anti-correlation | Redundant features that carry the same information |
| 3 | Duplicates | Same entity appearing multiple times in the dataframe |
| 4 | Missing values correlation | Two features always missing together |
| 5 | Missing values (MCAR/MAR/MNAR) | Different mechanisms behind the absence of data |
| 6 | Target leakage | Features that encode post-outcome information |
| 7 | Inconsistent categorical labels | "NY", "New York", "new york" for the same concept |
| 8 | Implicit hierarchies | Taxonomy stored as flat strings ("zone>neighborhood") |
| 9 | Unit inconsistencies | Same feature in sqft vs sqm across rows |
| 10 | Data drift | Distribution shifts pre/post policy change |
| 11 | Rare categories / long tail | Many categories with very few observations |
| 12 | Mixed types | Numeric stored as strings, codes mixed with free text |
| 13 | Invalid / impossible values | Negative ages, future dates, zero where impossible |
| 14 | Truncated / capped values | Operational limits (e.g., max loan cap at 293K) |
| 15 | Row-wise misalignment | Values shifted into wrong columns |
| 16 | Sparsity | High-dimensional binary flags that are mostly zero |
| 17 | Time misalignment / leakage | Future information leaking into training features |
| 18 | Business process artifacts | ETL batch IDs, workflow codes that correlate with target |
| 19 | Sampling bias / selection bias | Different data-generating regimes mixed together |

</details>

In [ ]:
import os
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

### Imports

In [ ]:
import tabulate
import os
import seaborn as sns
from datetime import datetime
from matplotlib.patches import Circle
import matplotlib.pyplot as plt
plt.style.use('bmh')
import numpy as np
import pandas as pd
pd.options.display.max_columns = 200
pd.options.display.max_colwidth = 250

def show(df, n_rows=5):
    print(tabulate.tabulate(df.head(n_rows), headers='keys', tablefmt='psql'))

## <a id="dataset-overview"></a> Dataset Overview

<div style="max-height:300px; overflow:auto; border:1px solid #ddd; padding:6px">

| Dataset | Task | Issues file (rows x cols) | Main issues |
|---|---|---:|---|
| Adult Income | Binary classification (<span style="color:#c0392b;font-weight:bold">`class`</span>: `<=50K` / `>50K`) | `adult_income_issues.csv` (50,317 x 36) | Missingness, leakage, duplicates, mixed types, invalid values, rare/new categories, sparse indicators, process artifacts |
| Ames Housing | Regression (<span style="color:#c0392b;font-weight:bold">`sale_price`</span>) | `ames_housing_issues.csv` (1,576 x 84) | Truncation/capping, multiple rows per entity |
| Retail Panel | Time-aware store-level forecasting (<span style="color:#c0392b;font-weight:bold">`sales`</span>) | `retail_panel_issues.csv` (22,800 x 9) | Data drift, time misalignment, process artifacts |

</div>

### Adult Income (`adult_income_issues.csv`)
Person-level application/income dataset with added operational and quality issues. Target is binary: <span style="color:#c0392b;font-weight:bold">`class`</span> is either `<=50K` or `>50k`. 

<div style="max-height:300px; overflow:auto; border:1px solid #ddd; padding:6px">

| person_id | age | occupation | hours_per_week | native_country | <span style="color:#c0392b;font-weight:bold">class</span> | split | post_adjudication_risk_code | duplicate_application_flag | record_written_at |
|---|---:|---|---:|---|---|---|---:|---:|---|
| 1 | 25 | Machine-op-inspct | 40 | United-States | <span style="color:#c0392b"><=50K</span> | train | 0.005 | 0 | 2026-01-20 03:21:27 |
| 2 | 38 | Farming-fishing | 50 | United-States | <span style="color:#c0392b"><=50K</span> | train | 0.021 | 0 | 2026-02-08 22:54:51 |
| 3 | 28 | Protective-serv | 40 | US | <span style="color:#c0392b">>50K</span> | test | 0.992 | 0 | 2026-01-23 11:09:29 |

</div>

### Ames Housing (`ames_housing_issues.csv`)
Property-level housing dataset for sale price prediction, with panel-style duplicates and capped derived values. Target is continuous <span style="color:#c0392b;font-weight:bold">`sale_price`</span>.

<div style="max-height:300px; overflow:auto; border:1px solid #ddd; padding:6px">

| property_id | mszoning | neighborhood | lotarea | grlivarea | <span style="color:#c0392b;font-weight:bold">sale_price</span> | reported_max_loan | record_extract_month |
|---|---|---|---:|---:|---:|---:|---|
| 1 | RL | CollgCr | 8450 | 1710 | <span style="color:#c0392b">208500.0</span> | 187650.0 | 2024-01 |
| 2 | RL | Veenker | 9600 | 1262 | <span style="color:#c0392b">181500.0</span> | 163350.0 | 2024-01 |
| 3 | RL | CollgCr | 11250 | 1786 | <span style="color:#c0392b">223500.0</span> | 201150.0 | 2024-01 |

</div>

### Retail Panel (`retail_panel_issues.csv`)
Daily store panel with promotion/inventory context and injected temporal/business-process issues. Target is continuous <span style="color:#c0392b;font-weight:bold">`sales`</span>.

<div style="max-height:300px; overflow:auto; border:1px solid #ddd; padding:6px">

| store_id | date | inventory_units | promotion_discount_pct | <span style="color:#c0392b;font-weight:bold">sales</span> | inventory_after_restock | manual_override_flag | workflow_route_code |
|---:|---|---:|---:|---:|---:|---:|---|
| 1 | 2023-01-01 | 561 | 7.59 | <span style="color:#c0392b">143</span> | 702.0 | 0 | AUTO_PASS |
| 1 | 2023-01-02 | 702 | 11.79 | <span style="color:#c0392b">139</span> | 423.0 | 0 | AUTO_PASS |
| 1 | 2023-01-03 | 423 | 1.43 | <span style="color:#c0392b">88</span> | 448.0 | 0 | AUTO_PASS |

</div>

In [ ]:
adult = pd.read_csv("day1/generated/adult_income_issues.csv")
ames = pd.read_csv("day1/generated/ames_housing_issues.csv")
retail = pd.read_csv("day1/generated/retail_panel_issues.csv")

## Data Issues

### <a id="duplicates"></a> 1. Duplicates

Duplicates can artificially inflate sample size, bias learned patterns, and create train/test contamination if the same entity appears in both sets.



#### See the Issue
*Manual inspection -- let's look at the data and spot what's wrong.*

In [ ]:
print("2 rows completely equal, no need to keep both: \n")

show(adult.loc[adult.duplicated(keep=False)].sort_values('person_id').head(2))

In [ ]:
dup_id = adult.groupby('person_id').nunique().max(axis=1).rename('nunique').reset_index().query('nunique > 1').person_id.iloc[0]
print(f"Same person_id, different values in other columns: {', '.join(adult.query(f"person_id == {dup_id}").nunique(axis=0).rename('n_rows').reset_index().query('n_rows > 1')['index'].tolist())}")

show(adult.query(f"person_id == {dup_id}"))

#### Spot It Automatically
*Programmatic detection -- how would you find this at scale?*

In [ ]:
print(f"Number of rows in dataset: {len(adult)}. Number of unique rows in dataset: {len(adult.drop_duplicates())}.\n")

print(f"Number of rows in dataset: {len(adult)}. Number of unique ids in dataset: {adult['person_id'].nunique(dropna=True)}.\n\n")

---
> **Do It Yourself**
>
> 1. Check if the Ames Housing dataset has duplicate `property_id` values. How many?
> 2. Are they exact duplicates (all columns equal) or conflicting duplicates (same ID, different values)?
> 3. What would happen if you trained a model without removing these duplicates?
>
> *Hint: use `ames.groupby("property_id").nunique()` to check for conflicting rows.*
---
######

### <a id="impossible-values"></a> 2. Impossible Values

Based on the column name (and sometimes on a description) we might know what values to expect. If a variable is "salary" or "age" you won't expect negative values for instance.

#### See the Issue
*Manual inspection -- let's look at the data and spot what's wrong.*

In [ ]:
def count_and_show(df, mask, cols, label, n=5):
    mask_sum = mask.sum()
    print(f"{label}: {int(mask_sum)} rows")
    if mask_sum > 0:
        show(df.loc[mask, cols].head(n), n)

In [ ]:
adult_age = pd.to_numeric(adult["age"], errors="coerce")
adult_bad_age = (adult_age > 120) | (adult_age < 0)

adult_cols = ["person_id", "age", "record_written_at"] if "record_written_at" in adult.columns else ["person_id", "age"]
count_and_show(adult, adult_bad_age, adult_cols, "Adult - invalid age (<0 or >120)", n=10)


In [ ]:
lot_num = pd.to_numeric(ames["lot_area"], errors="coerce")
count_and_show(ames, lot_num == 0, ["property_id", "lot_area"], "Ames - lot area == 0")

garage_num = pd.to_numeric(ames["garage_area"], errors="coerce")
count_and_show(ames, garage_num < 0, ["property_id", "garage_area"], "Ames - negative garage area")

year_num = pd.to_numeric(ames["year_built"], errors="coerce")
current_year = pd.Timestamp.today().year
count_and_show(ames, year_num > current_year, ["property_id", "year_built"], "Ames - future construction year")

In [ ]:
disc_num = pd.to_numeric(retail["promotion_discount_pct"], errors="coerce")
inv_num = pd.to_numeric(retail["inventory_units"], errors="coerce")

count_and_show(
    retail,
    (disc_num > 100) | (disc_num < 0),
    ["store_id", "date", "promotion_discount_pct"],
    "Retail - invalid discount (<0 or >100)"
)

count_and_show(
    retail,
    inv_num < 0,
    ["store_id", "date", "inventory_units"],
    "Retail - negative inventory"
)

#### Spot It Automatically
*Programmatic detection -- how would you find this at scale?*

In [ ]:
plt.style.use('tableau-colorblind10')

fig = plt.figure(figsize=(10, 5))
adult_age.plot(kind='hist', bins=40, title='Distribution of age in adult dataset')
ax1 = plt.gca()
ax1.axvline(0, color='red', linestyle='--', linewidth=2, label='Age < 0 (impossible)')
ax1.axvline(120, color='orange', linestyle='--', linewidth=2, label='Age > 120 (unrealistic)')
ax1.legend()
ax1.text(0.02, 0.95, f"Negative ages: {(adult_age < 0).sum()} rows\nAges > 120: {(adult_age > 120).sum()} rows", 
         transform=ax1.transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig = plt.figure(figsize=(10, 5))
garage_num.plot(kind='hist', bins=40, title='Distribution of garage area in ames dataset')
ax2 = plt.gca()
ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='Garage area < 0 (impossible)')
ax2.legend()
ax2.text(0.02, 0.95, f"Negative garage areas: {(garage_num < 0).sum()} rows", 
         transform=ax2.transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

---
> **Do It Yourself**
>
> 1. Write a generic function that checks for impossible values given a column and a valid range (e.g., `age` in `[0, 120]`).
> 2. Apply it to at least 3 numeric columns across the datasets.
> 3. For each, decide: would you drop the row, clip the value, or set it to NaN?
>
> *Hint: `pd.to_numeric(col, errors="coerce")` handles mixed types gracefully.*
---

######

### <a id="mixed-types"></a> 3. Mixed Types

Sometimes the raw dtype is misleading: a numeric field is stored as a string, a datetime arrives as text, or a numeric-looking column is actually an ID/code. Before fixing anything, it helps to profile how parseable each column is as numeric, datetime, or boolean.

In [ ]:
type_profile_rows = []
bool_tokens = {"true", "false", "yes", "no", "y", "n", "t", "f", "0", "1"}
dateish_pattern = r"[-/:]|\\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\\b"

for col in adult.columns:
    s = adult[col]
    raw = s.astype("string").str.strip()
    non_missing = raw.dropna()

    numeric_ratio = pd.to_numeric(non_missing, errors="coerce").notna().mean() if len(non_missing) else 0.0
    dateish_mask = non_missing.str.contains(dateish_pattern, case=False, regex=True, na=False)
    datetime_ratio = pd.to_datetime(non_missing.where(dateish_mask), errors="coerce", format="mixed").notna().mean() if len(non_missing) else 0.0
    boolean_ratio = non_missing.str.lower().isin(bool_tokens).mean() if len(non_missing) else 0.0
    avg_len = non_missing.str.len().mean() if len(non_missing) else 0.0

    note = ""
    lower = col.lower()
    if s.dtype.kind not in "biufc" and numeric_ratio >= 0.9:
        note = "stored as string, mostly numeric-like"
    elif s.dtype.kind not in "biufc" and datetime_ratio >= 0.9:
        note = "stored as string, mostly datetime-like"
    elif s.dtype.kind not in "biufc" and boolean_ratio >= 0.95 and non_missing.nunique() <= 5:
        note = "stored as string, mostly boolean-like"
    elif s.dtype.kind in "biufc" and (lower.endswith("_id") or lower.endswith("_key") or "code" in lower):
        note = "stored as numeric, but name suggests id/code"
    elif s.dtype.kind in "biufc" and s.nunique(dropna=True) <= 5:
        note = "stored as numeric, but low-cardinality"

    type_profile_rows.append({
        "column": col,
        "raw_dtype": str(s.dtype),
        "n_missing": int(s.isna().sum()),
        "n_unique": int(s.nunique(dropna=True)),
        "n_unique_including_na": int(raw.fillna("__MISSING__").nunique()),
        "pct_parseable_numeric": round(100 * float(numeric_ratio), 2),
        "pct_parseable_datetime": round(100 * float(datetime_ratio), 2),
        "pct_parseable_boolean": round(100 * float(boolean_ratio), 2),
        "avg_string_length": round(float(avg_len), 1) if len(non_missing) else 0.0,
        "note_or_warning": note,
        "example_values": non_missing.head(3).tolist(),
    })

mixed_type_profile = pd.DataFrame(type_profile_rows)
mixed_type_profile = mixed_type_profile.sort_values(["note_or_warning", "pct_parseable_numeric", "pct_parseable_datetime"], ascending=[False, False, False])

show(mixed_type_profile.loc[mixed_type_profile["note_or_warning"] != ""], n_rows=20)

In [ ]:
adult_hours_raw = adult["hours_per_week"].astype("string").str.strip()
adult_hours_num = pd.to_numeric(adult_hours_raw, errors="coerce")
adult_hours_mixed = adult_hours_raw.notna() & adult_hours_num.isna()

adult_age_raw = adult["age"].astype("string").str.strip()
adult_age_num = pd.to_numeric(adult_age_raw, errors="coerce")
adult_age_mixed = adult_age_raw.notna() & adult_age_num.isna()

suspect_rows = adult_hours_mixed | adult_age_mixed
suspect_cols = ["person_id", "age", "hours_per_week"]
if "record_written_at" in adult.columns:
    suspect_cols.append("record_written_at")

print(f"Adult - rows where numeric fields are actually strings: {int(suspect_rows.sum())} rows")
show(adult.loc[suspect_rows, suspect_cols], n_rows=10)

print("hours_per_week offending tokens:")
print(adult.loc[adult_hours_mixed, "hours_per_week"].value_counts().head(10))

print("\nage offending tokens:")
print(adult.loc[adult_age_mixed, "age"].value_counts().head(10))

date_like_cols = [c for c in ["record_written_at", "db_loaded_at_utc"] if c in adult.columns]
if date_like_cols:
    date_preview = adult[date_like_cols].head(5).copy()
    for c in date_like_cols:
        date_preview[f"{c}_parsed"] = pd.to_datetime(adult[c].head(5), errors="coerce", format="mixed")
    print("\nString columns that are parseable as datetimes:")
    show(date_preview, n_rows=5)


### <a id="row-wise-column-misalignment"></a> 4. Row-wise Column Misalignment

It corrupts the record structure itself (values in wrong columns).

In [ ]:
bad_rows = adult[
    pd.to_numeric(adult["age"], errors="coerce").isna() |
    adult["occupation"].astype(str).str.fullmatch(r"\d+")
]

display(bad_rows[["person_id", "age", "workclass", "occupation", "relationship"]].head(20))

### <a id="inconsistent-categorical-labels"></a> 5. Inconsistent Categorical Labels

Same concept appears under multiple strings, so you split signal and inflate cardinality.

In [ ]:
adult["native_country"].value_counts(dropna=False).head(30)

variants = [
    "United-States", "US", "U.S.A.", "united states",
    "Mexico", "MX", "mexico", "MEX",
    "India", "IN", "india", "Republic of India"
]

show(adult.loc[adult["native_country"].isin(variants), ["person_id", "native_country"]].drop_duplicates('native_country', ignore_index=True).head(20), 10)

---
> **Do It Yourself**
>
> 1. Find all variants of "United-States" in the `native_country` column. How many unique spellings are there?
> 2. Write a mapping dictionary to normalize them into a single value.
> 3. Can you think of an automated approach (e.g., fuzzy matching) that would scale to hundreds of categories?
---

### <a id="missing-values"></a> 6. Missing Values

Simply put: less information does not help any model.

#### Missingness Mechanisms (Quick Recap)

- **MCAR (Missing Completely At Random)**  
  Missingness is unrelated to both observed and unobserved values.  
  Example: random file corruption removes some entries.  
  Practical clue: rows with missing vs non-missing values look similar on other variables.

- **MAR (Missing At Random)**  
  Missingness depends on other **observed** columns, not on the missing value itself (conditional on observed data).  
  Example: income is missing more often for younger people.  
  Practical clue: a missingness flag can be predicted from other observed features.

- **MNAR (Missing Not At Random)**  
  Missingness depends on the **missing value itself** (or unobserved factors tied to it).  
  Example: very high income is less likely to be reported.  
  Practical clue: cannot be confirmed from the dataset alone; requires assumptions, domain knowledge, sensitivity checks, or external data.

In [ ]:
ames.loc[ames["lot_area"].isna(), ["property_id", "lot_area"]].head(10)

In [ ]:
ames.loc[ames["garage_area"].isna(), ["property_id", "year_built", "garage_area"]].head(10)

In [ ]:
show(ames.assign(missing_sale_price=ames["sale_price"].isna()).groupby("missing_sale_price")["reported_max_loan"].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ames.query("sale_price.isna()").reported_max_loan.plot(kind='kde', label='missing sale price')
ames.query("sale_price.notna()").reported_max_loan.plot(kind='kde', label='populated sale price')
ax.set_xlabel("reported_max_loan")
ax.set_title("Distribution of reported max loan by missingness of sale price")
ax.legend();

You can see that when `sale_price` is missing, `reported_max_loan` tends to be higher. If there is a pattern in the missingness of a feature then we can exclude that feature being MCAR. Unfortunately, without further knowledge, we can't state if the feature is MNAR, for instance if, given the same `reported_max_loan` value, lower `sale_price` are more likely to be missing. 

MCAR: missingness is random, so imputation mostly helps keep sample size and reduce variance.

MAR: missingness depends on other observed features, so you can recover information by modeling it, and ignoring it can bias the model because the missing group is systematically different.


---
> **Do It Yourself**
>
> 1. For each dataset, compute the percentage of missing values per column. Which columns are most affected?
> 2. Pick one column with missing values. Can you determine whether the missingness is MCAR, MAR, or MNAR? What evidence supports your conclusion?
> 3. Try comparing the distribution of another feature (e.g., `age`) for rows where `occupation` is missing vs present. What do you observe?
>
> *Hint: `df.isna().mean().sort_values(ascending=False)` gives a quick missingness overview.*
---

### <a id="perfect-correlation-anti-correlation"></a> 7. Perfect Correlation / Anti-Correlation

Highly correlated features are often **redundant**: they add little new signal, increase complexity, and can destabilize interpretation without improving predictive power. 

Pearson correlation captures linear pairwise relationships only. Tools like Mutual Information can reveal non-linear dependencies.

In [ ]:
(ames.select_dtypes('number').corr().unstack().rename('correlation').reset_index()
 .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2'})
 .query('feature_1 != feature_2')
 .assign(abs_correlation=lambda df: df.correlation.abs())
 .assign(pair_of_features=lambda df: df.apply(lambda row: tuple(sorted([row.feature_1, row.feature_2])), axis=1))
 .drop_duplicates('pair_of_features', ignore_index=True)
 .query("abs_correlation>0.99"))


In [ ]:
plt.style.use('fivethirtyeight')
corr_matrix = ames.select_dtypes("number").corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(20, 20))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=0.5,
    square=True,
    ax=ax,
)

ax.set_title("Correlation Matrix - Ames", fontsize=18, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
EPS = 1e-3
rows = []

def pick(df, *cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

def check_formula(df, name, lhs_col, rhs_series):
    lhs = pd.to_numeric(df[lhs_col], errors="coerce")
    rhs = pd.to_numeric(rhs_series, errors="coerce")
    mask = lhs.notna() & rhs.notna()
    if mask.sum() == 0:
        return
    resid = (lhs[mask] - rhs[mask]).abs()
    rows.append({
        "formula": name,
        "rows_used": int(mask.sum()),
        "max_abs_resid": float(resid.max()),
        "mean_abs_resid": float(resid.mean()),
        "is_linear_combo": bool(float(resid.max()) <= EPS),
    })

a = pick(ames, "gr_liv_area", "grlivarea")
b = pick(ames, "total_bsmt_sf", "totalbsmtsf")
g = pick(ames, "garage_area", "garagearea")

if a and "gr_liv_area_m2" in ames.columns:
    check_formula(
        ames,
        "gr_liv_area_m2 = 0.09290304 * gr_liv_area",
        "gr_liv_area_m2",
        pd.to_numeric(ames[a], errors="coerce") * 0.09290304,
    )

if a and b and g and "total_enclosed_area_sqft" in ames.columns:
    check_formula(
        ames,
        "total_enclosed_area_sqft = gr_liv_area + total_bsmt_sf + garage_area",
        "total_enclosed_area_sqft",
        pd.to_numeric(ames[a], errors="coerce")
        + pd.to_numeric(ames[b], errors="coerce")
        + pd.to_numeric(ames[g], errors="coerce"),
    )

if "total_enclosed_area_m2" in ames.columns and "total_enclosed_area_sqft" in ames.columns:
    check_formula(
        ames,
        "total_enclosed_area_m2 = 0.09290304 * total_enclosed_area_sqft",
        "total_enclosed_area_m2",
        pd.to_numeric(ames["total_enclosed_area_sqft"], errors="coerce") * 0.09290304,
    )

if "retail" in globals() and {"full_price_pct", "promotion_discount_pct"}.issubset(retail.columns):
    check_formula(
        retail,
        "full_price_pct = 100 - promotion_discount_pct",
        "full_price_pct",
        100 - pd.to_numeric(retail["promotion_discount_pct"], errors="coerce"),
    )

pd.DataFrame(rows).sort_values("max_abs_resid").reset_index(drop=True)


---
> **Do It Yourself**
>
> 1. Find all pairs of features in the Adult dataset with correlation above 0.9 (or below -0.9).
> 2. For one such pair, would you drop one? Which one, and why?
> 3. Can you find two features that have low Pearson correlation but are clearly related? (Hint: try non-linear relationships.)
---

### <a id="missing-values-correlation"></a> 8. Missing Values Correlation

Two non-correlated variables might be very correlated when considering their **missingness patterns**. If two features are always missing together, that's a strong signal about data collection or process issues.

In [ ]:
null_corr_matrix = retail.isna().astype(int).corr()

mask = np.triu(np.ones_like(null_corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(20, 20))
sns.heatmap(
    null_corr_matrix,
    mask=mask,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=0.5,
    square=True,
    ax=ax,
)

ax.set_title("Null Correlation Matrix - Retail", fontsize=18, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
retail.query("(full_price_pct.isna() & promotion_discount_pct.notna()) | (full_price_pct.notna() & promotion_discount_pct.isna())")

### <a id="rare-categories"></a> 9. Rare Categories

In [ ]:
# Adult: rare categories in occupation
freq = adult["occupation"].value_counts(dropna=False)
rare = freq[freq <= 20]  # threshold you can tune

print("Rare categories (<=20 rows):")
print(rare)


### <a id="unit-inconsistencies"></a> 10. Unit Inconsistencies

In [ ]:
lot_col = "lot_area" 

ratio = ames["lot_area_reported"] / ames[lot_col]

likely_sqm = ratio.between(0.09, 0.095, inclusive="both")
likely_sqft = ratio.between(0.99, 1.01, inclusive="both")

show(
    ames.loc[likely_sqm, ["property_id", lot_col, "lot_area_reported"]].head(20)
)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ames["lot_area_reported"].plot(kind="kde", ax=ax, label="reported lot area")

x_min = ames["lot_area_reported"].min() - 3000
x_max = ames["lot_area_reported"].max()
ax.set_xlim(x_min, x_max)

x = ames["lot_area_reported"].dropna().to_numpy()
counts, bins = np.histogram(x, bins=120)
centers = (bins[:-1] + bins[1:]) / 2

left_mask = centers < np.nanmedian(x)
right_mask = ~left_mask

sqm_mode_x = centers[left_mask][np.argmax(counts[left_mask])]
sqft_mode_x = centers[right_mask][np.argmax(counts[right_mask])]

kde_line = ax.lines[0]
xk, yk = kde_line.get_xdata(), kde_line.get_ydata()
sqm_mode_y = np.interp(sqm_mode_x, xk, yk)
sqft_mode_y = np.interp(sqft_mode_x, xk, yk)

ymax = max(yk)
r = ymax * 0.08
ax.add_patch(Circle((sqm_mode_x, sqm_mode_y), r, fill=False, ec="royalblue", lw=2.5))
ax.add_patch(Circle((sqft_mode_x, sqft_mode_y), r, fill=False, ec="crimson", lw=2.5))

ax.annotate("square meters", xy=(sqm_mode_x, sqm_mode_y), xytext=(sqm_mode_x - 2500, sqm_mode_y + ymax*0.12),
            arrowprops=dict(arrowstyle="->", color="royalblue", lw=1.8),
            color="royalblue", fontsize=11, fontweight="bold")
ax.annotate("square feet", xy=(sqft_mode_x, sqft_mode_y), xytext=(sqft_mode_x + 1500, sqft_mode_y + ymax*0.10),
            arrowprops=dict(arrowstyle="->", color="crimson", lw=1.8),
            color="crimson", fontsize=11, fontweight="bold")

ax.legend()
fig.suptitle("Density of reported lot area - Ames", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()



### <a id="truncated-capped-values"></a> 11. Truncated / Capped Values

In [ ]:
loan_col = "reported_max_loan"
price_col = "sale_price" if "sale_price" in ames.columns else "saleprice"
cap_value = 293500

# 1) Show capped rows
display(
    ames.loc[ames[loan_col] == cap_value, ["property_id", price_col, loan_col]].head(20)
)



fig, ax = plt.subplots(figsize=(9, 4))
#ames[loan_col].plot(kind="kde", ax=ax)
ames[loan_col].plot(kind="hist", bins=50,ax=ax, alpha=0.4)
ax.axvline(cap_value, color="red", linestyle="--", linewidth=2, label=f"cap = {cap_value}")
ax.set_title("Ames - reported_max_loan (capped)")
ax.legend()
plt.show()

### <a id="implicit-hierarchies"></a> 12. Implicit Hierarchies

Not an issue per se, yet: when hierarchical information is stored as one flat category, models cannot explicitly learn effects at each level of granularity; splitting levels usually improves interpretability, parameter sharing, and robustness.

Some examples:

- Geography: country > state > city > zip
- Product taxonomy: department > category > subcategory > SKU
- Organization: company > division > team > role
- Manufacturing: plant > line > machine > sensor

In [ ]:
# Ames example: taxonomy stored as a flat string "zone>neighborhood"
display(ames[["property_taxonomy_flat"]].head(10))

# Split the hierarchy into separate levels
taxonomy = ames["property_taxonomy_flat"].str.split(">", n=1, expand=True)
taxonomy.columns = ["zone_level", "neighborhood_level"]

display(taxonomy.head(10))

# Attach to original rows for clarity
display(
    pd.concat([ames[["property_taxonomy_flat"]], taxonomy], axis=1).head(15)
)


### <a id="unseen-categories-at-testing-time"></a> 13. Unseen Categories at Testing Time

In [ ]:
# Next issue: New categories at test/inference time (#1)

# In this dataset, split column is already present (train/test)
train_occ = set(adult.loc[adult["split"] == "train", "occupation"].dropna().astype(str))
test_occ = set(adult.loc[adult["split"] == "test", "occupation"].dropna().astype(str))

unseen_in_test = sorted(test_occ - train_occ)

print("Unseen categories in test (occupation):")
print(unseen_in_test)

display(
    adult.loc[
        (adult["split"] == "test") & (adult["occupation"].astype(str).isin(unseen_in_test)),
        ["person_id", "split", "occupation"]
    ].head(20)
)


---
> **Do It Yourself**
>
> 1. For each categorical column in the Adult dataset, compute `set(test[col]) - set(train[col])`. Which columns have unseen categories?
> 2. How would a one-hot encoder handle an unseen category at test time? What about a label encoder?
> 3. Propose a strategy for handling unseen categories that is robust at deployment time.
---

### <a id="sparsity"></a> 14. Sparsity

Very high-dimensional sparse indicators add noise and complexity, making models harder to train, interpret, and maintain.

In [ ]:
sparse_cols = [
    c for c in adult.columns
    if c.endswith("_flag")
]

# Show sparsity (fraction of zeros) for each flag
sparsity = (
    (adult[sparse_cols] == 0).mean()
    .sort_values(ascending=False)
    .rename("zero_fraction")
    .to_frame()
)

display(sparsity.head(20))

# Quick view of these columns
display(adult[sparse_cols].head(10))


### <a id="data-drift"></a> 15. Data Drift

When feature distributions change over time, models trained on older periods can degrade in production.

In [ ]:
ret = retail.copy()
ret["date"] = pd.to_datetime(ret["date"])
cutoff = pd.Timestamp("2023-10-01")

ret["period"] = np.where(ret["date"] < cutoff, "pre_policy", "post_policy")

# Simple comparison of a key feature/target
display(
    ret.groupby("period")["sales"].describe()
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.kdeplot(data=ret, x="sales", hue="period", common_norm=False, ax=ax)
ax.set_title("Retail sales distribution: pre vs post period")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ret.index, ret["macro_cost_index"], label="macro_cost_index")
ax.set_xlabel("Index")
ax.set_ylabel("macro_cost_index")
ax.set_title("Distribution of legacy channel share - Retail")
plt.tight_layout()
plt.show()

### <a id="time-misalignment-leakage"></a> 16. Time Misalignment / Leakage

In [ ]:
ret = retail.copy()
ret["date"] = pd.to_datetime(ret["date"])
ret = ret.sort_values(["store_id", "date"])

ret["inventory_t_plus_1"] = ret.groupby("store_id")["inventory_units"].shift(-1)

# Show that inventory_after_restock is effectively future information
display(
    ret[["store_id", "date", "inventory_units", "inventory_after_restock", "inventory_t_plus_1"]].head(20)
)

# quick check
(
    ret["inventory_after_restock"].fillna(-9999)
    == ret["inventory_t_plus_1"].fillna(-9999)
).mean()


### <a id="target-leakage"></a> 17. Target Leakage

In [ ]:
adult_tmp = adult.copy()
adult_tmp["target_binary"] = adult_tmp["class"].astype(str).str.contains(">50").astype(int)

# Compare leaked feature by target class
display(
    adult_tmp.groupby("class")["post_adjudication_risk_code"].describe()
)

# Correlation with target (should be very high)
adult_tmp[["post_adjudication_risk_code", "target_binary"]].corr()


---
> **Do It Yourself**
>
> 1. Identify at least 2 columns in the Adult dataset that could be leaking target information. What makes you suspect them?
> 2. For one suspected leaky feature, compute its correlation with the target. Is it suspiciously high?
> 3. Ask yourself: "Would this feature be available at the time I need to make a prediction in production?" If not, it's likely leakage.
---

### <a id="business-process-artifacts"></a> 18. Business Process Artifacts

In [ ]:
# Retail
display(
    retail.groupby("workflow_route_code")["sales"].describe()
)

# Adult (ETL/process artifacts)
adult[["db_source_table", "db_etl_batch_id", "dataset_schema_version", "extract_country_code"]].head(10)

# Always-missing feature: every single value is NaN
print(f"\nretail['store_type'] — non-null count: {retail['store_type'].notna().sum()} / {len(retail)}")


### <a id="sampling-bias-selection-bias"></a> 19. Sampling Bias / Selection Bias

In [ ]:
display(
    adult.groupby("dgp_regime")[["education_num", "capital_gain"]].describe()
)

display(
    adult.groupby("dgp_regime")["class"].value_counts(normalize=True).rename("share").reset_index()
)


---

## Final Checkpoint

Before moving to the preprocessing notebook, make sure you can answer these questions:

- [ ] I can identify at least 5 different types of data issues from looking at a new dataset
- [ ] I understand the difference between MCAR, MAR, and MNAR
- [ ] I know what target leakage is and how to detect it
- [ ] I can distinguish between process/metadata columns and real features
- [ ] I understand why unseen categories at test time are a problem
- [ ] I can spot correlated features and explain why redundancy matters
- [ ] I know when data drift or time misalignment could invalidate a model

*To check off items, double-click this cell and change `[ ]` to `[x]`.*